In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:

# List the 6 monthly files
files = [
    "../data/CRMLSSold202512.csv",
    "../data/CRMLSSold202601.csv",
    "../data/CRMLSSold202602.csv",
    "../data/CRMLSSold202603.csv",
    "../data/CRMLSSold202604.csv",
    "../data/CRMLSSold202605.csv",
]

In [3]:

df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print(f"Total rows loaded: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(df.head())

C:\Users\linji\AppData\Local\Temp\ipykernel_20128\269367619.py:1: DtypeWarning: Columns (0: WaterfrontYN, 1: PostalCode) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


Total rows loaded: 124404
Total columns: 78
  BuyerAgentAOR ListAgentAOR          Flooring ViewYN WaterfrontYN BasementYN  \
0   ContraCosta  ContraCosta  Carpet,Tile,Wood    NaN          NaN        NaN   
1   Mlslistings  Mlslistings               NaN  False          NaN        NaN   
2      SanDiego     SanDiego       Carpet,Wood   True          NaN        NaN   
3   Mlslistings  Mlslistings               NaN  False          NaN        NaN   
4   TriCounties  TriCounties               NaN   True          NaN        NaN   

  PoolPrivateYN  OriginalListPrice  ListingKey  \
0         False          1998000.0  1150041639   
1           NaN                NaN  1150038872   
2         False          2214421.0  1150038683   
3           NaN          1200000.0  1150038607   
4          True             3300.0  1150038314   

                     ListAgentEmail  ... LotSizeDimensions  LotSizeArea  \
0           teresa@teresahooper.com  ...               NaN      10080.0   
1           homes@

In [4]:
df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
]

print(f"Rows after filtering: {len(df)}")

Rows after filtering: 61727


In [5]:
# Keep only columns relevant for modeling
cols_to_keep = [
    "ClosePrice", "CloseDate",
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "Stories",
    "GarageSpaces", "ParkingTotal", "AttachedGarageYN",
    "FireplaceYN", "FireplacesTotal", "PoolPrivateYN",
    "ViewYN", "WaterfrontYN",
    "AssociationFee", "TaxAnnualAmount",
    "ListPrice", "OriginalListPrice", "DaysOnMarket",
    "City", "PostalCode", "CountyOrParish",
    "HighSchoolDistrict", "Latitude", "Longitude",
    "NewConstructionYN"
]

df = df[cols_to_keep].copy()
print(f"Columns kept: {len(df.columns)}")
print(df.dtypes)

Columns kept: 28
ClosePrice               float64
CloseDate                    str
LivingArea               float64
BedroomsTotal            float64
BathroomsTotalInteger    float64
LotSizeSquareFeet        float64
YearBuilt                float64
Stories                  float64
GarageSpaces             float64
ParkingTotal             float64
AttachedGarageYN          object
FireplaceYN               object
FireplacesTotal          float64
PoolPrivateYN             object
ViewYN                    object
WaterfrontYN              object
AssociationFee           float64
TaxAnnualAmount          float64
ListPrice                float64
OriginalListPrice        float64
DaysOnMarket               int64
City                         str
PostalCode                object
CountyOrParish               str
HighSchoolDistrict           str
Latitude                 float64
Longitude                float64
NewConstructionYN         object
dtype: object


In [6]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({
    "Missing Count": missing,
    "Missing %": missing_pct
}).sort_values("Missing %", ascending=False)

print(missing_summary[missing_summary["Missing Count"] > 0])

                       Missing Count  Missing %
TaxAnnualAmount                61727      100.0
FireplacesTotal                61727      100.0
WaterfrontYN                   61691       99.9
AssociationFee                 17616       28.5
HighSchoolDistrict             16629       26.9
AttachedGarageYN                7527       12.2
Stories                         6493       10.5
ViewYN                          5295        8.6
PoolPrivateYN                   4833        7.8
NewConstructionYN               4686        7.6
GarageSpaces                    2353        3.8
LotSizeSquareFeet               1081        1.8
OriginalListPrice                136        0.2
FireplaceYN                       58        0.1
YearBuilt                         36        0.1
BathroomsTotalInteger              1        0.0
LivingArea                        30        0.0
City                              14        0.0
PostalCode                         1        0.0
Latitude                           9    

In [7]:
# Drop rows where target variable is missing
df = df.dropna(subset=["ClosePrice"])

# Numerical columns — fill with median
num_cols = ["LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
            "LotSizeSquareFeet", "YearBuilt", "GarageSpaces",
            "ParkingTotal", "FireplacesTotal", "AssociationFee",
            "TaxAnnualAmount", "ListPrice", "OriginalListPrice",
            "DaysOnMarket", "Stories", "Latitude", "Longitude"]

for col in num_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Boolean columns — fill with 0 (assume No if unknown)
bool_cols = ["AttachedGarageYN", "FireplaceYN", "PoolPrivateYN",
             "ViewYN", "WaterfrontYN", "NewConstructionYN"]

for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].fillna(False)

# Categorical columns — fill with "Unknown"
cat_cols = ["City", "PostalCode", "CountyOrParish", "HighSchoolDistrict"]
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

print("Missing values after handling:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Missing values after handling:
FireplacesTotal    61727
TaxAnnualAmount    61727
dtype: int64


In [8]:
# Convert boolean columns to 0/1
for col in bool_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

# Encode City, PostalCode, CountyOrParish, HighSchoolDistrict
# Using label encoding (assigns a number to each category)
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
for col in cat_cols:
    if col in df.columns:
        df[col + "_encoded"] = le.fit_transform(df[col].astype(str))

print("Encoded columns added:")
print([c for c in df.columns if "_encoded" in c])

Encoded columns added:
['City_encoded', 'PostalCode_encoded', 'CountyOrParish_encoded', 'HighSchoolDistrict_encoded']


In [9]:
# Convert CloseDate to datetime
df["CloseDate"] = pd.to_datetime(df["CloseDate"])

# Most recent month = test set (May 2026)
# Remaining months = training set
test_month = df["CloseDate"].max().to_period("M")

test_df  = df[df["CloseDate"].dt.to_period("M") == test_month].copy()
train_df = df[df["CloseDate"].dt.to_period("M") != test_month].copy()

print(f"Test set (most recent month): {len(test_df)} rows — {test_month}")
print(f"Train set: {len(train_df)} rows")

Test set (most recent month): 12024 rows — 2026-05
Train set: 49703 rows


In [10]:
output_path = r"C:\Users\linji\OneDrive\Desktop\IDXExchange\ca-price-prediction\data\cleaned_data.csv"
df.to_csv(output_path, index=False)
print(f"Cleaned data saved: {df.shape[0]} rows, {df.shape[1]} columns")

Cleaned data saved: 61727 rows, 32 columns
